**Table of contents**<a id='toc0_'></a>    
- [Photoswitching fingerprints: 2 fluorophores](#toc1_)    
  - [Data generation](#toc1_1_)    
  - [Reading and fitting the data](#toc1_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Photoswitching fingerprints: 2 fluorophores](#toc0_)

In [1]:
import glob

import numpy as np
import pandas as pd

import fluopy.fitting as fit
import fluopy.fluorophores as fl
import fluopy.routines as rt
import fluopy.transitions as tr

%load_ext autoreload
%autoreload 2

saving_at = r"D:\python_output\Chapter_I\0_4_multi_f_PFA\2F"

## <a id='toc1_1_'></a>[Data generation](#toc0_)

In [2]:
def prepare_transition_set(distance):
    fluorophores = fl.construct_fluorophores(
        name="cy5_dna", count=2, distance=distance, shape="square"
    )
    fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)
    transitions = fluorophore_system.load_transitions(
        bleaching=True,
        summarize=True,
        energy_transfer=True,
        **rt.PARAMS_DSTORM,
    )
    transition_set = tr.TransitionSet(transitions, fluorophore_system)
    transition_set.finalize()

    return transition_set

In [3]:
rng = np.random.default_rng(1)
distances = [3, 6, 9, 18]
for distance in distances:
    transition_set = prepare_transition_set(distance)
    _, _, _ = rt.fingerprint_analysis(
        transition_set,
        batch_size=40,
        batches=1,
        filepath=saving_at,
        filename=f"{distance}nm",
        seed=rng,
        use_memmap=r"C:\Users\vie43sq\Desktop\Simulations\memmaps\run_1",
    )

Floating point precision error warning:
 The smallest safe increment is 5.68e-14.
 Everything drawn below this number might be rounded to zero
 when approaching the time limit of this simulation.
 Using the highest possible rate which occurs for example in state combination [1, 1]
 gives a probability of 1.39e-02 for a smaller increment to be drawn.
Floating point precision error warning:
 The smallest safe increment is 5.68e-14.
 Everything drawn below this number might be rounded to zero
 when approaching the time limit of this simulation.
 Using the highest possible rate which occurs for example in state combination [1, 1]
 gives a probability of 1.39e-02 for a smaller increment to be drawn.
Floating point precision error warning:
 The smallest safe increment is 5.68e-14.
 Everything drawn below this number might be rounded to zero
 when approaching the time limit of this simulation.
 Using the highest possible rate which occurs for example in state combination [1, 1]
 gives a proba

## <a id='toc1_2_'></a>[Reading and fitting the data](#toc0_)

In [ ]:
distances = [3, 6, 9, 18]
identifiers = [f"{distance}nm" for distance in distances]
fingerprints_all = []
x = np.linspace(0, 300, 300001)
rng = np.random.default_rng(1)
for i, id in enumerate(identifiers):
    fingerprints_all.append(
        pd.Series(np.zeros(300001), np.round(x, decimals=12), dtype=np.int32)
    )
    for file in glob.glob(saving_at + "/*"):
        if file.endswith(".parquet") and id in file:
            fingerprints_all[i] += pd.read_parquet(file).sum(axis=1)
    fingerprint = fingerprints_all[i].cumsum() / fingerprints_all[i].sum()
    y = fingerprint.values
    result = fit.ps_fingerprint_cdf_fit_2f(
        x, y, maxiter=2000, popsize=50, disp=False, rng=rng
    )
    np.save(saving_at + "\\" + rf"fitting_parameters\\{id}.npy", result.x)

c:\Users\vie43sq\uv_envs\phd_main\Lib\site-packages\scipy\optimize\_differentiable_functions.py:317: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(self.x - self.x_prev, self.g - self.g_prev)
c:\Users\vie43sq\uv_envs\phd_main\Lib\site-packages\scipy\optimize\_differentiable_functions.py:317: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(self.x - self.x_prev, self.g - self.g_prev)
c:\Users\vie43sq\uv_envs\phd_main\Lib\site-packages\scipy\optimize\_differentiable_functions.py:317: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the 

: 